In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:

columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income"
]

df = pd.read_csv(
    "adult_income_raw.csv",
    sep=", ",
    engine="python",
    header=None,
    names=columns,
    na_values="?"
)
print("the dataset is loaded successfully ")

the dataset is loaded successfully 


In [3]:
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,"""39",State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,"<=50K"""
1,"""50",Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,"<=50K"""
2,"""38",Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,"<=50K"""
3,"""53",Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,"<=50K"""
4,"""28",Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,"<=50K"""


In [4]:
numerical_cols = df.select_dtypes(include =[ "int64" , "float64"]).columns

In [5]:
numerical_cols

Index(['fnlwgt', 'education_num', 'capital_gain', 'capital_loss',
       'hours_per_week'],
      dtype='object')

In [6]:
num_median_cols = df[numerical_cols].columns

We are including all numerical columns in this median applicable columns list because according to the missing value analysis we did the best option for filling the missing values is median 

In [7]:
categtorical_cols = df.select_dtypes(include = "object").columns

In [8]:
cat_mode_cols = df[["workclass" , "occupation" ,"native_country"]].columns

In [9]:
cat_unknown_cols = [col for col in categtorical_cols if col not in cat_mode_cols]

In [10]:
num_median_pipeline = Pipeline(steps = [
    ("imputer" , SimpleImputer(strategy = "median")),
    ("scale" , StandardScaler())
])

In [11]:
cat_mode_pipeline = Pipeline(steps = [
    ("imputer" , SimpleImputer(strategy = "most_frequent" )),
    ("Encoder" , OneHotEncoder(handle_unknown="ignore"))
        
])

In [12]:
cat_unknown_pipeline = Pipeline(steps = [
    ("imputer" , SimpleImputer(strategy="constant" , fill_value="Unknown")),
    ("Encoder" , OneHotEncoder(handle_unknown="ignore"))
])

In [13]:
transformer = ColumnTransformer(transformers=[
    ("num_median" , num_median_pipeline , num_median_cols),
    ("cat_mode" , cat_mode_pipeline , cat_mode_cols),
    ("cat_unknown" , cat_unknown_pipeline , cat_unknown_cols)
])

In [14]:
final_pipeline = Pipeline(steps=[
    ("transformer" , transformer)
])

In [15]:
df_preprocessed = final_pipeline.fit_transform(df)

In [16]:
df_preprocessed

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 488430 stored elements and shape (32562, 189)>

In [17]:
df_preprocessed = pd.DataFrame(df_preprocessed.toarray())

In [18]:
df_preprocessed.head()

,0,1,2,3,4,5,6,7,8,9,...,179,180,181,182,183,184,185,186,187,188
0,-1.063624,1.134757,0.148460,-0.216656,-0.035429,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,-1.008719,1.134757,-0.145918,-0.216656,-2.222186,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.245086,-0.420065,-0.145918,-0.216656,-0.035429,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.425811,-1.197476,-0.145918,-0.216656,-0.035429,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.408200,1.134757,-0.145918,-0.216656,-0.035429,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


FINAL PREPROCESSING PIPELINE EXPLANATION:
---------------------------------------------
The preprocessing pipeline has successfully transformed the raw dataset into a structured numerical format suitable for machine learning models.

Key steps completed in this preprocessing workflow:

- Missing values were handled using appropriate imputation strategies for numerical and categorical features.
                                                          
- Numerical features were scaled using StandardScaler after median-based imputation.
                                                          
- Categorical features were processed using two strategies:
  - Mode imputation followed by One-Hot Encoding.
  - Constant imputation ("Unknown") followed by One-Hot Encoding.
                                                          
- A ColumnTransformer was used to apply the appropriate pipeline to each group of columns.
                                                          
- The entire preprocessing workflow was encapsulated in a unified Pipeline for reproducibility and scalability.

The final output is a transformed feature matrix where all features are numeric and ready for machine learning algorithms.

PREPROCESSING PIPELINE STRUCTURE
------------------------------------

The preprocessing workflow follows a modular pipeline structure:

Raw Dataset
     │
     ▼
Column Selection
     │
     ▼
ColumnTransformer
 ├── Numerical Pipeline
 │     ├── Median Imputation
 │     └── Standard Scaling
 │
 ├── Categorical Mode Pipeline
 │     ├── Most Frequent Imputation
 │     └── One-Hot Encoding
 │
 └── Categorical Unknown Pipeline
       ├── Constant Imputation ("Unknown")
       └── One-Hot Encoding
     │
     ▼
Final Transformed Feature Matrix

In [19]:
joblib.dump(final_pipeline, "preprocessing_pipeline.pkl")

['preprocessing_pipeline.pkl']

In [26]:
df_processed = pd.DataFrame(df_preprocessed)
df_preprocessed.to_csv( "C:\\Data_preprocessing_engine\\Data\\processed\\adult_preprocessed.csv", index=False)